# Naive Bayes — Implementations

Gaussian Naive Bayes twice more: once on tensors, once through sklearn. The estimator is closed form — two reductions per class and a broadcast in log space — so both lanes reproduce the NumPy parameters to machine precision. The only trap worth naming is the smoothing epsilon: conventions that look identical can differ in what variance they scale it by.

## 08_gaussian_naive_bayes

One Gaussian per class per feature, glued together by the independence assumption and read off in log space.

### torch

The same closed-form estimator on float64 tensors: `fit` is two reductions per class, prediction one broadcast subtraction in log space. **What torch adds:** batched log-density evaluation that runs unchanged on a GPU — nothing is trained, so autograd never enters; the exercise is keeping every convention (ddof=0, the epsilon rule) identical to the NumPy lane.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Everything is closed form: no gradients, just tensor reductions done in log space.
# 2. unbiased=False makes torch.var match np.var's default ddof=0 population variance.
# 3. Broadcast X as (n,1,d) against means (K,d); sum the log-densities over axis 2 last.
# 4. Keep the epsilon rule identical: var_smoothing times max(overall feature var, 1.0).


class GaussianNBScratch:
    """Gaussian Naive Bayes on float64 tensors — the same closed-form estimator."""

    def __init__(self, var_smoothing: float = 1e-9):
        self.var_smoothing = var_smoothing
        self.classes_ = None
        self.mean_ = None       # (n_classes, n_features) tensor
        self.var_ = None        # (n_classes, n_features) tensor
        self.class_log_prior_ = None

    def fit(self, X, y):
        """Per-class means and population variances — two reductions per class."""
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        y = np.asarray(y)
        self.classes_ = np.unique(y)

        # Same smoothing rule as the NumPy lane, on the whole data set.
        epsilon = self.var_smoothing * max(float(Xt.var(dim=0, unbiased=False).max()), 1.0)

        means, variances, counts = [], [], []
        for c in self.classes_:
            X_c = Xt[torch.as_tensor(y == c)]
            counts.append(X_c.shape[0])
            means.append(X_c.mean(dim=0))
            variances.append(X_c.var(dim=0, unbiased=False) + epsilon)
        self.mean_ = torch.stack(means)
        self.var_ = torch.stack(variances)
        self.class_log_prior_ = torch.log(
            torch.as_tensor(counts, dtype=torch.float64) / len(y))
        return self

    def _joint_log_likelihood(self, X):
        """log P(y=k) + sum_j log N(x_j | mu_kj, var_kj), shape (n, K)."""
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        diff = Xt[:, None, :] - self.mean_[None, :, :]
        log_density = -0.5 * (torch.log(2.0 * torch.pi * self.var_)[None, :, :]
                              + diff ** 2 / self.var_[None, :, :])
        return self.class_log_prior_[None, :] + log_density.sum(dim=2)

    def predict(self, X):
        """argmax over classes of the joint log-likelihood."""
        jll = self._joint_log_likelihood(X)
        return self.classes_[jll.argmax(dim=1).numpy()]

    def score(self, X, y):
        return float(np.mean(self.predict(X) == np.asarray(y)))


In [ ]:
# exports: mean, var, log_prior, jll, preds_eq, acc_eq
_rng_eq = np.random.default_rng(8)
_mus_eq = np.array([[0.0, 0.0, 0.0], [3.5, 1.5, -1.5], [1.5, 4.0, 2.5]])
_counts_eq = [40, 30, 20]
X_eq = np.vstack([_mus_eq[k] + _rng_eq.normal(size=(_counts_eq[k], 3)) for k in range(3)])
y_eq = np.repeat(np.arange(3), _counts_eq)
_Xq_eq = np.array([[0.0, 0.0, 0.0], [3.5, 1.5, -1.5], [1.5, 4.0, 2.5],
                   [1.8, 2.0, 0.5], [-1.0, 2.0, 1.0]])

_gnb_eq = GaussianNBScratch(var_smoothing=1e-9).fit(X_eq, y_eq)
mean = np.asarray(_gnb_eq.mean_)
var = np.asarray(_gnb_eq.var_)
log_prior = np.asarray(_gnb_eq.class_log_prior_)
jll = np.asarray(_gnb_eq._joint_log_likelihood(_Xq_eq))
preds_eq = _gnb_eq.predict(_Xq_eq).astype(int)
acc_eq = _gnb_eq.score(X_eq, y_eq)
print("accuracy:", round(acc_eq, 4), "priors:", np.round(np.exp(log_prior), 3))


In [ ]:
# A point sitting exactly on a class mean is at that Gaussian's density peak:
# its joint log-likelihood is the log prior minus 0.5*sum(log(2*pi*var)).
_jll_mu = _gnb_eq._joint_log_likelihood(np.asarray(_gnb_eq.mean_))
_peak = _gnb_eq.class_log_prior_ - 0.5 * torch.log(2.0 * torch.pi * _gnb_eq.var_).sum(dim=1)
assert torch.allclose(torch.diagonal(_jll_mu), _peak, atol=1e-9), \
    "each class mean sits on its own density peak"

# Translation invariance: shift every feature by a constant and refit — means
# shift with the data, variances (and so the smoothing epsilon) do not move.
_gnb_shift = GaussianNBScratch(var_smoothing=1e-9).fit(X_eq + 7.5, y_eq)
_jll_shift = _gnb_shift._joint_log_likelihood(_Xq_eq + 7.5)
assert torch.allclose(_jll_shift, torch.as_tensor(jll), atol=1e-8), \
    "Gaussian NB is invariant under a constant feature shift"

# Priors are the exact class frequencies, and predict is argmax of the joint.
assert torch.allclose(torch.exp(_gnb_eq.class_log_prior_),
                      torch.tensor([40, 30, 20], dtype=torch.float64) / 90.0)
assert np.array_equal(preds_eq, _gnb_eq.classes_[np.argmax(jll, axis=1)])
assert acc_eq > 0.9, "three well-separated blobs should be almost fully recovered"


### library

sklearn's `GaussianNB` wrapped behind the notebook's names. **What the library adds:** the identical estimator plus `partial_fit` for streaming and `predict_log_proba`; its `var_smoothing` scales the largest overall feature variance with no `max(..., 1.0)` floor, so on this fixture (max variance > 1) every parameter and every joint log-likelihood matches the scratch lane to machine precision.

In [ ]:
import numpy as np
from sklearn.naive_bayes import GaussianNB

# hints:
# 1. sklearn smooths every variance by var_smoothing times the largest feature variance.
# 2. The scratch floors that reference at 1.0; sklearn does not — keep max var above 1.
# 3. theta_, var_ and class_prior_ map onto mean_, var_ and exp(class_log_prior_).
# 4. predict_log_proba is the joint minus its logsumexp; compare joints, not posteriors.


class GaussianNBScratch:
    """sklearn's GaussianNB behind the notebook's attribute names. Both use
    ddof=0 variances and epsilon = var_smoothing * max feature variance, so on
    data whose largest feature variance exceeds 1 (where the scratch lane's
    max(..., 1.0) floor is inactive) every learned parameter matches exactly."""

    def __init__(self, var_smoothing: float = 1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self._model = GaussianNB(var_smoothing=self.var_smoothing).fit(X, y)
        self.classes_ = self._model.classes_
        self.mean_ = self._model.theta_
        self.var_ = self._model.var_
        self.class_log_prior_ = np.log(self._model.class_prior_)
        return self

    def _joint_log_likelihood(self, X):
        return self._model._joint_log_likelihood(np.asarray(X, dtype=float))

    def predict(self, X):
        return self._model.predict(np.asarray(X, dtype=float))

    def score(self, X, y):
        return float(np.mean(self.predict(X) == np.asarray(y)))


In [ ]:
# exports: mean, var, log_prior, jll, preds_eq, acc_eq
_rng_eq = np.random.default_rng(8)
_mus_eq = np.array([[0.0, 0.0, 0.0], [3.5, 1.5, -1.5], [1.5, 4.0, 2.5]])
_counts_eq = [40, 30, 20]
X_eq = np.vstack([_mus_eq[k] + _rng_eq.normal(size=(_counts_eq[k], 3)) for k in range(3)])
y_eq = np.repeat(np.arange(3), _counts_eq)
_Xq_eq = np.array([[0.0, 0.0, 0.0], [3.5, 1.5, -1.5], [1.5, 4.0, 2.5],
                   [1.8, 2.0, 0.5], [-1.0, 2.0, 1.0]])

_gnb_eq = GaussianNBScratch(var_smoothing=1e-9).fit(X_eq, y_eq)
mean = np.asarray(_gnb_eq.mean_)
var = np.asarray(_gnb_eq.var_)
log_prior = np.asarray(_gnb_eq.class_log_prior_)
jll = np.asarray(_gnb_eq._joint_log_likelihood(_Xq_eq))
preds_eq = _gnb_eq.predict(_Xq_eq).astype(int)
acc_eq = _gnb_eq.score(X_eq, y_eq)
print("accuracy:", round(acc_eq, 4), "priors:", np.round(np.exp(log_prior), 3))


In [ ]:
# The convention translation is exact on this data: sklearn's smoothed
# variance is the per-class population variance plus var_smoothing times the
# largest overall feature variance — the notebook's rule, verbatim.
_eps = 1e-9 * float(np.var(X_eq, axis=0).max())
for _k, _c in enumerate(_gnb_eq.classes_):
    assert np.allclose(_gnb_eq.var_[_k], X_eq[y_eq == _c].var(axis=0) + _eps, atol=1e-12), \
        "sklearn's var_ is population variance + epsilon, same as the scratch lane"

# predict_log_proba is exactly the joint log-likelihood, normalised by its
# row-wise logsumexp — nothing beyond Bayes' rule happens inside.
_m = jll.max(axis=1, keepdims=True)
_lse = np.log(np.exp(jll - _m).sum(axis=1, keepdims=True)) + _m
assert np.allclose(jll - _lse, _gnb_eq._model.predict_log_proba(_Xq_eq), atol=1e-10)

assert np.allclose(np.exp(_gnb_eq.class_log_prior_), np.array([40, 30, 20]) / 90.0)
assert np.array_equal(preds_eq, _gnb_eq.classes_[np.argmax(jll, axis=1)])
assert acc_eq > 0.9
